# Appendix C: Subspace Products Retrieved

Printed source span verified before authoring: Appendix C is listed as printed pages 597-602. The PDF text layer contains printed pages 597-601 on PDF pages 613-617; printed page 602 appears to be an omitted blank page before Appendix D. This notebook is an original executable reference for the appendix's central idea: once the geometric product is available, the familiar subspace products can be recovered by selecting grades.

This is a powerful implementation idea. A library can implement the geometric product as the primitive bilinear product, then define outer product, contractions, and scalar product as grade-filtered views of that product for homogeneous inputs. The idea is compact, but it has a boundary: if inputs are mixed-grade multivectors, a single target grade is no longer enough. You decompose by grade first, apply the homogeneous rule pairwise, and then sum the pieces.


## Translation Guide

For a grade `r` blade `A_r` and a grade `s` blade `B_s`, the outer product is the grade `r+s` part of `A_r B_s`. The left contraction is the grade `s-r` part, with the understood zero result when `r > s`. The right contraction is the grade `r-s` part. The scalar product is the grade zero part.

The reason this works is not that the geometric product magically forgets the old products. It contains them. Multiplying two homogeneous blades produces a multivector whose grades encode how much of the product is spanning behavior, how much is metric overlap, and how much is scalar agreement. Grade projection retrieves the component with the desired geometric meaning.

The notebook uses the grade-selection rules as executable identities. Each direct product is compared with its grade-selected version. Then we test vector formulas that express outer product and contraction through the geometric product and grade involution.


## Route

We start with the verified source audit. Then we use one vector and one bivector to show a geometric product with both contraction and outer-product content. Next, we run a battery of homogeneous grade-selection checks. After that, we check the vector identities involving grade involution, build a small visual map of possible product grades, and finish by showing how mixed-grade inputs must be decomposed before applying the homogeneous formulas.


In [ ]:
from pathlib import Path
import json, math, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
PROJECT_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "utils" / "ga").exists(): PROJECT_ROOT = candidate; break
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
from utils.ga import Algebra
from utils.appendix_reference import (
    APPENDIX_ARTIFACT_ROOT, basis_blades, grade_selection_records, left_contract_from_gp,
    outer_from_gp, right_contract_from_gp, scalar_from_gp, source_span_records,
    write_csv_artifact, write_json_artifact,
)
pd.set_option("display.max_columns", 14)
APPENDIX_KEY = "appendix-c"
artifact_paths = {}
def remember(label, path): artifact_paths[label] = str(Path(path)); return Path(path)
print("project", PROJECT_ROOT)


In [ ]:
source_rows = source_span_records()
source_payload = {"pdf":"Geometric Algebra for Computer Science.pdf","verification_tools":["pdfinfo","pdftotext -layout"],"pdf_page_count":640,"note":"Printed pages are authoritative; omitted blank pages are listed explicitly.","records":source_rows}
remember("source_span_json", write_json_artifact(["source"], "source-span-audit.json", source_payload))
remember("source_span_csv", write_csv_artifact(["source"], "source-span-audit.csv", source_rows))
pd.DataFrame(source_rows)


## A Geometric Product With Multiple Grades

A vector times a bivector is a good first example because the geometric product can contain a vector part and a trivector part. The vector part is the contraction. The trivector part is the outer product. If the vector lies partly inside the plane of the bivector, the contraction is nonzero; if it also has a component outside the plane, the outer product is nonzero.

The example below uses `a = e1 + 2 e3` and `B = e1 wedge e2`. The `e1` component contracts into the plane; the `e3` component spans volume with it.


In [ ]:
alg = Algebra([1, 1, 1], names=["e1", "e2", "e3"])
e1, e2, e3 = alg.basis()
a = e1 + 2 * e3
B = e1.wedge(e2)
product = a.gp(B)
first_records = grade_selection_records("a", a, "B", B)
remember("first_grade_selection_json", write_json_artifact([APPENDIX_KEY,"checks"], "first-grade-selection.json", first_records))
{"aB":repr(product), "grade_1":repr(product.grade(1)), "grade_3":repr(product.grade(3)), "records":first_records}


## Homogeneous Product Checks

The grade approach is cleanest for homogeneous inputs. The table below checks several pairs: vector-vector, vector-bivector, bivector-vector, bivector-bivector, scalar-vector, and trivector-vector. Each row compares a direct product implementation with the grade-selected geometric product.

This is also a useful test pattern for a product-table implementation. If the geometric product is correct but grade selection is wrong, these rows fail. If grade selection is correct but the direct outer or contraction product is wrong, the rows fail. The redundancy is helpful because Appendix C is precisely about aligning these definitions.


In [ ]:
e12 = e1.wedge(e2); e23 = e2.wedge(e3); I = alg.pseudoscalar()
pairs = [("e1",e1,"e2",e2),("a",a,"e12",e12),("e12",e12,"a",a),("e12",e12,"e23",e23),("2",alg.scalar(2),"e3",e3),("I",I,"e1",e1)]
selection_rows = []
for left_name, left, right_name, right in pairs:
    selection_rows.extend(grade_selection_records(left_name, left, right_name, right))
assert all(row["matches_direct"] for row in selection_rows)
remember("homogeneous_grade_selection_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "homogeneous-grade-selection.csv", selection_rows))
pd.DataFrame(selection_rows)


## Vector Identities From The Geometric Product

Appendix C also records vector identities that retrieve outer product and contraction using grade involution. For a vector `a` and multivector `B`, the sum of `aB` with the grade-involuted companion extracts the outer behavior, while the difference extracts the contraction behavior. These identities are especially useful when the geometric product is the primitive operation in an implementation.

The next cell tests the identities on a mixed-grade `B`. This is a compact way to check that grade involution signs are implemented correctly.


In [ ]:
mixed_B = e2 + 3 * e12 + 0.5 * I
outer_via_gp = (a.gp(mixed_B) + mixed_B.grade_involution().gp(a)) * 0.5
contract_via_gp = (a.gp(mixed_B) - mixed_B.grade_involution().gp(a)) * 0.5
direct_outer = a.wedge(mixed_B)
direct_contract = a.left_contract(mixed_B)
vector_identity_payload = {"a":repr(a),"B":repr(mixed_B),"outer_via_gp":repr(outer_via_gp),"direct_outer":repr(direct_outer),"outer_matches":outer_via_gp.almost_equal(direct_outer),"contract_via_gp":repr(contract_via_gp),"direct_contract":repr(direct_contract),"contract_matches":contract_via_gp.almost_equal(direct_contract)}
assert vector_identity_payload["outer_matches"] and vector_identity_payload["contract_matches"]
remember("vector_identities_json", write_json_artifact([APPENDIX_KEY,"checks"], "vector-identities.json", vector_identity_payload))
vector_identity_payload


## Grade Support Map

Grade selection is easier to remember when the possible grades are visible. For two homogeneous inputs of grades `r` and `s`, the outer product asks for the high grade `r+s`. Contractions ask for grade differences. The geometric product may contain a ladder of grades between those extremes, depending on overlaps and dimension.

The following heatmap marks the outer target grade for all pairs in a four-dimensional algebra. Cells outside the algebra dimension are impossible and are left blank in the saved data as `None`. This is not a proof; it is a navigational map for implementation and debugging.


In [ ]:
n = 4
grade_map = np.full((n+1,n+1), np.nan)
map_rows = []
for r in range(n+1):
    for s in range(n+1):
        target = r+s if r+s <= n else None
        map_rows.append({"grade_r":r,"grade_s":s,"outer_target":target,"left_contract_target":s-r,"right_contract_target":r-s})
        if target is not None: grade_map[r,s] = target
fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(grade_map, cmap="viridis", vmin=0, vmax=n)
ax.set_xlabel("grade s"); ax.set_ylabel("grade r"); ax.set_title("Outer-product target grade r+s")
ax.set_xticks(range(n+1)); ax.set_yticks(range(n+1))
for r in range(n+1):
    for s in range(n+1):
        label = "-" if np.isnan(grade_map[r,s]) else str(int(grade_map[r,s]))
        ax.text(s, r, label, ha="center", va="center", color="white" if not np.isnan(grade_map[r,s]) and grade_map[r,s] > 2 else "black")
fig.colorbar(im, ax=ax, shrink=0.8); fig.tight_layout()
map_path = remember("grade_support_png", APPENDIX_ARTIFACT_ROOT / APPENDIX_KEY / "figures" / "grade-support-map.png")
map_path.parent.mkdir(parents=True, exist_ok=True); fig.savefig(map_path, dpi=160); plt.close(fig)
remember("grade_support_csv", write_csv_artifact([APPENDIX_KEY,"tables"], "grade-support-map.csv", map_rows))
pd.DataFrame(map_rows).head(12)


## Grade Selection As An Oracle

The following implementation oracle loops over all basis blades in a three-dimensional Euclidean algebra. For every homogeneous pair it checks four identities: outer product, left contraction, right contraction, and scalar product. This is the smallest practical expression of Appendix C as a unit test.

A real implementation would also test non-orthogonal metrics, sparse coefficient storage, and tolerance behavior. Still, the basis-blade test is powerful because most product implementations reduce to basis-blade multiplication plus bilinear extension.


In [ ]:
blades = basis_blades(alg)
oracle_failures = []
for left_name, left in blades.items():
    for right_name, right in blades.items():
        checks_for_pair = [("outer",outer_from_gp(left,right),left.wedge(right)),("left",left_contract_from_gp(left,right),left.left_contract(right)),("right",right_contract_from_gp(left,right),left.right_contract(right)),("scalar",scalar_from_gp(left,right),left.scalar_product(right))]
        for name, recovered, direct in checks_for_pair:
            if not recovered.almost_equal(direct): oracle_failures.append({"left":left_name,"right":right_name,"product":name,"recovered":repr(recovered),"direct":repr(direct)})
oracle_payload = {"failure_count":len(oracle_failures),"tested_pairs":len(blades)**2,"failures":oracle_failures[:10]}
assert not oracle_failures
remember("grade_selection_oracle_json", write_json_artifact([APPENDIX_KEY,"checks"], "grade-selection-oracle.json", oracle_payload))
oracle_payload


## Mixed-Grade Inputs, Pitfalls, And Takeaways

The homogeneous formulas do not license a shortcut for mixed multivectors. If `M` has scalar, vector, and bivector parts while `N` has vector and bivector parts, a product such as the outer product is the sum over all homogeneous grade pairs. There is no single grade `r+s` because there is no single `r` or `s`.

The main pitfall is applying a homogeneous grade formula to a mixed-grade multivector without decomposing it. The second pitfall is forgetting impossible-grade zeros. A requested grade below zero or above the algebra dimension is simply zero in this setting. Implement the geometric product carefully. Implement grade projection carefully. Define derived products through grade projection for homogeneous components. Use direct product implementations as tests against grade-selected recovery, and use grade-selected recovery as tests against direct implementations. Appendix C is short because the idea is sharp: the geometric product is a container, and grades are the retrieval keys.


In [ ]:
def homogeneous_parts(mv):
    return {grade: mv.grade(grade) for grade in range(mv.algebra.dimension + 1) if mv.grade(grade).terms}
M = alg.scalar(2) + e1 + 0.5 * e12
N = e2 + e23
mixed_outer = alg.scalar(0); pair_terms = []
for r, M_r in homogeneous_parts(M).items():
    for s, N_s in homogeneous_parts(N).items():
        term = outer_from_gp(M_r, N_s)
        pair_terms.append({"r":r,"s":s,"term":repr(term)})
        mixed_outer = mixed_outer + term
direct_mixed_outer = M.wedge(N)
mixed_payload = {"M":repr(M),"N":repr(N),"pair_terms":pair_terms,"recovered_outer":repr(mixed_outer),"direct_outer":repr(direct_mixed_outer),"matches":mixed_outer.almost_equal(direct_mixed_outer)}
assert mixed_payload["matches"]
remember("mixed_grade_json", write_json_artifact([APPENDIX_KEY,"checks"], "mixed-grade-decomposition.json", mixed_payload))
mixed_payload


## Reference Note: Reading Grade Selectors

A grade selector should be read as a typed query against the geometric product. The input grades state what kind of answer is meaningful, and the selected grade states which geometric relationship is being requested. The high-grade query asks what new span is created. The grade-difference queries ask what part survives as metric overlap from the left or from the right. The scalar query asks for same-grade agreement. When a selector asks for a grade that cannot exist, the right answer is zero, not an exception. That policy keeps formulas composable and makes mixed-grade decomposition straightforward: split, query each homogeneous pair, and sum only the pieces that exist.


In [ ]:
assert all(row["matches_direct"] for row in selection_rows)
assert not oracle_failures
assert mixed_payload["matches"]
sanity_payload = {"appendix":"C", "artifact_count_before_sanity":len(artifact_paths), "artifacts":artifact_paths}
sanity_path = remember("final_sanity", write_json_artifact([APPENDIX_KEY,"checks"], "final-sanity.json", sanity_payload))
missing = [str(path) for path in map(Path, artifact_paths.values()) if not Path(path).exists()]
assert not missing, missing
print(json.dumps({"appendix":"C", "artifact_count":len(artifact_paths), "sanity":str(sanity_path)}, indent=2))
